# Config

In [44]:
import pandas as pd
import openpyxl
import re
from config import RAW_DATA_DIR, PROCESSED_DATA_DIR
pd.set_option('future.no_silent_downcasting', True)

# Read data

In [45]:
file_path = RAW_DATA_DIR / "FCTAES_279691_TOTAL_20251006.xlsx"

labels_df = pd.read_excel(file_path, sheet_name="Labels")
variables_df = pd.read_excel(file_path, sheet_name="Variables")
codes_df = pd.read_excel(file_path, sheet_name="Codes")

print("Labels",labels_df.shape)
print("Variables",variables_df.shape)
print("Codes",codes_df.shape)

Labels (1813, 514)
Variables (515, 4)
Codes (2442, 3)


# Clean data

## Manage wrong headers

In [46]:
# In codes_df, the first row is actually the header, so we need to clean that up
codes_df = codes_df.drop(index=0).reset_index(drop=True)
codes_df.columns = ["Variable", "Code", "Label"]
print("Cleaned Codes", codes_df.shape)
codes_df.head()

Cleaned Codes (2441, 3)


,Variable,Code,Label
0,SAMPLE,1,INTERNAL
1,NaN,2,EXTERNAL
2,NaN,3,TESTING
3,NaN,4,BBDD
4,SAMPLE_PROVIDER,1,NICEQUEST


In [47]:
# delete last row of variables_df
variables_df.drop(variables_df.index[-1], inplace=True)
variables_df.reset_index(drop=True, inplace=True)
variables_df.tail()

,Variable,Posición,Etiqueta,Nivel de medición
509,P91_cod2,510.0,How safe or unsafe would you feel using the ex...,Nominal
510,P91_cod3,511.0,How safe or unsafe would you feel using the ex...,Nominal
511,P91_cod4,512.0,How safe or unsafe would you feel using the ex...,Nominal
512,P91_cod5,513.0,How safe or unsafe would you feel using the ex...,Nominal
513,P91_cod6,514.0,How safe or unsafe would you feel using the ex...,Nominal


## Manage NaN


In [48]:
codes_df['Variable'] = codes_df['Variable'].ffill()
codes_df.head()

,Variable,Code,Label
0,SAMPLE,1,INTERNAL
1,SAMPLE,2,EXTERNAL
2,SAMPLE,3,TESTING
3,SAMPLE,4,BBDD
4,SAMPLE_PROVIDER,1,NICEQUEST


# Process Data (Get results)

## 1. Build code dictionary

In [49]:
code_dict = {}

for _, row in codes_df.iterrows():
    var = row["Variable"]
    code = row["Code"]
    label = row["Label"]

    code_dict.setdefault(var, {})[code] = label
code_dict

{'SAMPLE': {'1': 'INTERNAL', '2': 'EXTERNAL', '3': 'TESTING', '4': 'BBDD'},
 'SAMPLE_PROVIDER': {'1': 'NICEQUEST',
  '2': 'Partner_2',
  '3': 'Partner_3',
  '4': 'Partner_4',
  '5': 'Partner_5',
  '6': 'Partner_6',
  '7': 'Partner_7',
  '8': 'Partner_8',
  '9': 'Partner_9',
  '10': 'Partner_10',
  '11': 'Partner_11',
  '12': 'Partner_12',
  '13': 'Partner_13',
  '14': 'Partner_14',
  '15': 'Partner_15',
  '16': 'Partner_16',
  '17': 'Partner_17',
  '18': 'Partner_18',
  '19': 'Partner_19',
  '20': 'Partner_20',
  '21': 'Partner_21',
  '22': 'Partner_22',
  '23': 'Partner_23',
  '24': 'Partner_24',
  '25': 'Partner_25',
  '26': 'Partner_26',
  '27': 'Partner_27',
  '28': 'Partner_28',
  '29': 'Partner_29',
  '30': 'Partner_30',
  '31': 'Partner_31',
  '32': 'Partner_32',
  '33': 'Partner_33',
  '34': 'Partner_34',
  '35': 'Partner_35',
  '36': 'Partner_36',
  '37': 'Partner_37',
  '38': 'Partner_38',
  '39': 'Partner_39',
  '40': 'Partner_40',
  '41': 'Partner_41',
  '42': 'Partner_42',

## 2. Decode numeric values in Labels

In [50]:
decoded_df = labels_df.copy()

for column in decoded_df.columns:
    if column in code_dict:
        decoded_df[column] = decoded_df[column].map(
            lambda x: code_dict[column].get(x, x)
        )
decoded_df.head()

,key,numericalId,accessCount,startTime,endTime,duration,status,type,SAMPLE,CodPanelista,...,P84_cod3,P84_cod4,P84_cod5,P84_cod6,P91_cod1,P91_cod2,P91_cod3,P91_cod4,P91_cod5,P91_cod6
0,2fd1e655-a94d-4a91-a4fa-ed22b097ce62,684173892,1,2025-09-22 12:17:19,2025-09-22 12:32:20,900,end,complete,INTERNAL,01519d71455139b9,...,NaN,NaN,NaN,NaN,Protection against specific accidents (protect...,NaN,NaN,NaN,NaN,NaN
1,7da18c0c-caa8-4c5e-a5de-47f0805bb06e,684295712,1,2025-09-23 06:00:19,2025-09-23 06:15:05,886,end,complete,INTERNAL,02082c28ac6c270d,...,NaN,NaN,NaN,NaN,I don't like the product (don't need this prod...,NaN,NaN,NaN,NaN,NaN
2,f4fe2017-372a-4d5b-8a7c-6604cc560ec9,684182100,2,2025-09-22 13:36:32,2025-09-23 08:55:05,1293,end,complete,INTERNAL,0248c65078193632,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,224d885a-7942-417d-805d-82060fe5f206,684297408,3,2025-09-23 06:40:22,2025-09-23 06:49:51,578,end,complete,INTERNAL,0371c3d2680a5100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,e18389a7-a1d4-4697-a0af-e9776552cb2b,684296611,1,2025-09-23 06:21:05,2025-09-23 06:44:30,1405,end,complete,INTERNAL,03af3ddf6752205f,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Rename columns

In [51]:
# Creamos el mapa de nombres: {"Q1": "¿Cuál es su edad?"}
names_dict = dict(zip(variables_df['Variable'], variables_df['Etiqueta']))

# Renombramos todo el DataFrame de una vez
decoded_df.rename(columns=names_dict, inplace=True)

# Processing for R

## 1. Clean dictionaries

In [52]:
codes_df['Variable'] = codes_df['Variable'].astype(str).str.strip()
variables_df['Variable'] = variables_df['Variable'].astype(str).str.strip()


## 2. Treat Variables

In [53]:
cols_nominales = variables_df[variables_df['Nivel de medición'] == 'Nominal']['Variable'].unique()

for col in labels_df.columns:
    if col in cols_nominales:
        # Buscamos el mapeo para esta columna
        mapping = codes_df[codes_df['Variable'] == col]
        if not mapping.empty:
            map_dict = dict(zip(mapping['Variable'], mapping['Label']))
            labels_df[col] = labels_df[col].map(map_dict).fillna(labels_df[col])

In [54]:
cols_escala = variables_df[variables_df['Nivel de medición'] == 'Escala']['Variable'].unique()
for col in cols_escala:
    if col in labels_df.columns:
        labels_df[col] = pd.to_numeric(labels_df[col], errors='coerce')

## 3. Naming R

In [55]:
def clean_header_r(text):
    # Eliminar acentos
    text = text.replace('á', 'a').replace('é', 'e').replace('í', 'i').replace('ó', 'o').replace('ú', 'u')
    # Quitar caracteres especiales y dejar solo letras, números y espacios
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # Reemplazar espacios por puntos (estándar clásico de R)
    text = text.strip().replace(" ", ".")
    return text[:60] # Acortar para que sea manejable

# Crear el diccionario de nombres: { 'Q1': 'Nivel.de.satisfaccion...' }
name_mapping = dict(zip(variables_df['Variable'], variables_df['Etiqueta'].apply(clean_header_r)))
labels_df.rename(columns=name_mapping, inplace=True)

# Save data

In [59]:
# decoded_df.to_csv(PROCESSED_DATA_DIR/"survey_clean_powerbi.csv", index=False)

decoded_df.to_excel(PROCESSED_DATA_DIR/"IMPROVA_clean_for_powerbi.xlsx", index=False)


labels_df.to_csv(PROCESSED_DATA_DIR/'IMPROVA_clean_for_R.csv', index=False, encoding='utf-8')